# Lab 3: hybrid search, reranking, and what actually moved the number

**Time: about 75 minutes. Work in pairs.** One person types, the other checks.

**What you will build.** You take the index you built in Lab 1 and add three things to it, one at a
time: keyword search fused with the meaning search, a reranker that reorders the shortlist, and a
filter that drops documents that are no longer in force. After each one you measure the same 20
questions in the same way. At the end you can say which of the three earned its place and which one
did not, with a number for each.

**How the cells work.** Each step has up to three cells. The first shows you the real thing. The
second is marked `YOUR CALL` and is the only one you edit, where you write your prediction. The third
measures what really happens and prints your answer next to it. Every `YOUR CALL` cell already holds
a working value, so the notebook runs top to bottom before you touch it.

**Three words used throughout.** *Dense search* means search by meaning, using the vectors from Lab 1.
*BM25* is ordinary keyword search, which scores a document by how often your words appear in it,
weighted so that rare words count more. *nDCG@10* is a single score between 0 and 1 for a ranked list
of ten results: it is 1.0 when the best answers sit at the top, and it falls as good answers slide
down the list.

In [ ]:
%pip install -q pypdf python-docx python-pptx openpyxl rank_bm25

In [ ]:
# Setup for Databricks. Run this cell first: every later cell depends on it.
# It puts the lab files (src/ and data/) in this session's own folder, /tmp/freshcart, and puts
# src/ on the import path so `from freshcart import ...` works.
# If you opened this notebook from a Git folder of the repository, the files are copied from that
# folder. If you imported only this notebook, they are downloaded from GitHub.
import io, os, sys, pathlib, shutil, textwrap, urllib.request, zipfile

REPO_ZIP = "https://github.com/mogupta84/ai-engineer-data-labs/archive/refs/heads/main.zip"
ROOT = pathlib.Path("/tmp/freshcart")


def repo_folder():
    here = pathlib.Path(os.getcwd())          # on Databricks, the folder that holds this notebook
    for p in (here, *here.parents):
        if (p / "src" / "freshcart.py").is_file() and (p / "data" / "raw").is_dir():
            return p
    return None


ROOT.mkdir(parents=True, exist_ok=True)
found = repo_folder()
if found:
    for part in ("src", "data"):
        shutil.copytree(found / part, ROOT / part, dirs_exist_ok=True, copy_function=shutil.copyfile)
    print("lab files copied from your Git folder:", found)
else:
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(REPO_ZIP, timeout=120).read())) as z:
        for name in z.namelist():
            rel = name.split("/", 1)[1] if "/" in name else ""
            if rel.startswith(("src/", "data/")) and not name.endswith("/"):
                (ROOT / rel).parent.mkdir(parents=True, exist_ok=True)
                (ROOT / rel).write_bytes(z.read(name))
    print("lab files downloaded from GitHub")
for folder in ("out", "data/cache", "data/logs"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

RAW = ROOT / "data" / "raw"     # the seven original documents. Nothing ever writes here.
print("lab folder:", ROOT)
print("corpus files:", len(list(RAW.glob("*"))))

## Step 1. Watch the wrong document win

**What you will do.** Run the next cell. It rebuilds the Lab 1 corpus and puts one real question,
"how many refund claims can a customer make in thirty days", through meaning search alone. It prints
the top five pieces of text with the file each came from, the score, and whether that file has been
withdrawn.

**What you should see.** The corpus holds 30 pieces. The top hit is `refund_policy_v2_2022.pdf` with a
score near 0.94, and it is marked superseded, which means it is the withdrawn 2022 policy. The
current policy, `refund_policy_v3_2024.pdf`, sits below it.

**Why this step is here.** This is the Week 1 failure, and it is the reason this lab exists. The
search is not broken. The withdrawn policy genuinely is the closer match in wording, because the two
documents say almost the same thing. No change of embedding model fixes that, which is why the rest
of this lab is about everything except the model.

In [ ]:
# What each import gives you:
#   build_corpus  rebuilds the Lab 1 corpus: load, chunk, label, embed, index
#   load_golden   reads a judged question set from data/golden/
#   relevance     turns one question's grading rules into {piece id: grade}
#   Hybrid        meaning search and keyword search over the same pieces, merged
#   rerank        reorders a shortlist by scoring each candidate against the question
#   ndcg_at_k     one score for a ranked list: 1.0 when the best answers are on top
#   current_only  the recency filter from Lab 1, which drops withdrawn documents
from labkit import build_corpus, load_golden, relevance, table, banner
from retrieval import Hybrid, rerank, ndcg_at_k
from freshcart import current_only

corpus = build_corpus(False)     # False means the offline track, with no keys
print("pieces of text in the corpus:", len(corpus.chunks))

Q1 = "how many refund claims can a customer make in thirty days"
print("\nquestion:", Q1, "\n")
print(f"{'score':>6}  {'source':40s}  withdrawn")
for score, ch in corpus.index.search(Q1, k=5):
    print(f"{score:6.3f}  {ch.meta['source']:40s}  {'yes' if ch.meta.get('superseded') else 'no'}")

## Step 2. Put a number on where you are starting

**What you will do.** Load the 20 judged questions in `data/golden/retrieval_golden.json`. Each one
carries rules that say which text counts as a good answer and how good it is, on a scale of 0 to 3.
In the `YOUR CALL` cell, write your guess for the average nDCG@10 of meaning search across all 20.
Then run the reveal.

**What you should see.** A table of 20 rows and an average of 0.803. Five rows are marked `yes` in the
last column: those are the five questions that failed in Week 1. R17 scores 0.000, the worst in the
set, and R02 scores 0.31.

**Why this step is here.** Every claim you make later in this lab is a comparison against this number.
Without it you are left with "hybrid feels better", which is exactly what the room could not defend in
Week 1.

In [ ]:
# Each judged question carries rules, not a list of correct piece ids. A rule says
# "text from this file containing this phrase is worth this grade", so the judgements
# survive a change of chunk size, which would renumber every piece.
golden = load_golden("retrieval_golden.json")["queries"]
print(len(golden), "judged questions")
print("\nexample rules for", golden[0]["id"], ":")
for rule in golden[0]["rules"]:
    print("  grade", rule["grade"], "->", rule["source"], "containing", repr(rule["contains"]))

In [ ]:
# YOUR CALL: your guess for the average nDCG@10 of meaning search alone, 0 to 1
PREDICT_DENSE = 0.80

In [ ]:
import statistics

# One function, used for all four stages. Pass it a way of searching and it returns
# 20 scores, one per question. Only the search changes between stages, so the
# comparison at the end is fair.
def stage_scores(search_fn):
    out = []
    for item in golden:
        rel = relevance(item, corpus.chunks)          # {piece id: grade} for this question
        ids = [ch.id for _, ch in search_fn(item["query"])]   # what this stage returned, best first
        out.append(ndcg_at_k(ids, rel, k=10))         # score the top ten of that list
    return out

dense = stage_scores(lambda q: corpus.index.search(q, k=50))
rows = [{"id": it["id"], "query": it["query"][:44], "dense": round(s, 3),
         "week 1 failure": "yes" if it.get("week1_failure") else ""}
        for it, s in zip(golden, dense)]
print(table(rows, ["id", "query", "dense", "week 1 failure"]))
mean_dense = statistics.mean(dense)
print(f"\naverage nDCG@10, meaning search only: {mean_dense:.3f}")
print(f"you predicted {PREDICT_DENSE:.3f}, out by {abs(PREDICT_DENSE - mean_dense):.3f}")

## How nDCG@10 is calculated: a grade, a rank, and the best possible order

Every score in this lab is built from three things.

**1. Grade: how good is the chunk?** A person decided in advance, in the golden file, which text answers
each question, on a scale of 0 to 3. The grade is turned into points, so the best answer counts for far
more than a partly useful one:

| Grade | Meaning | Points |
|---|---|---|
| 3 | answers the question | 7 |
| 2 | relevant | 3 |
| 1 | related | 1 |
| 0 | anything else | 0 |

**2. Rank: where did it land?** Rank is the chunk's position in the list the search returned, 1 at the
top. A chunk keeps less of its points the lower it sits. The share it keeps is 1 / log2(rank + 1):

| Rank | Divided by | Share of points kept |
|---|---|---|
| 1 | 1.00 | 100% |
| 2 | 1.58 | 63% |
| 3 | 2.00 | 50% |
| 4 | 2.32 | 43% |
| 6 | 2.81 | 36% |
| 10 | 3.46 | 29% |

Why a logarithm and not the rank itself? Dividing by the rank would be too harsh: rank 10 would keep only
a tenth. The logarithm falls fast at the top and flattens lower down, which matches how people read a
list: the first few places matter most. It is the standard choice, so your numbers compare with other
teams'.

**3. Score: compare with the best possible order.** Add up the points of the top 10 results. Then work
out what the same graded chunks would earn in the perfect order, best first from rank 1. Divide the first
by the second:

    nDCG@10 = points earned / best possible points

So 1.0 always means "the best answers are on top", whether a question has one good answer or three, and
the 20 questions can be averaged into one number.

**Two quick examples.** A question with one grade-3 answer at rank 2 earns 7 x 63% = 4.42 out of a best
of 7: 0.631. Move that answer to rank 1 and it earns all 7: 1.000. That is R13 before and after
reranking. R02 earns 2.92 out of 9.39: 0.311. The next step prints that calculation for any question.

## Step 2b. Open up one question and watch its score being built

**What you will do.** Run the next cell. It takes one question, `R02` to start with, and prints every
step that turns the search results into its nDCG@10 score. Then change `QID` to `R17`, and after that
to `R13`, and run it again.

**How the score is built, in four steps.**

1. **The answer key.** `data/golden/retrieval_golden.json` holds the 20 questions. For each one, a
   person wrote rules of the form "any chunk from *this file* that contains *this phrase* gets *this
   grade*". Grades run from 3 (answers the question) through 2 (relevant) and 1 (related) to 0
   (everything else). The withdrawn 2022 policy is graded 0 on purpose: similar is not correct. Rules
   match text, not chunk numbers, so they still work if you chunk the documents differently.
2. **Grade every chunk.** `relevance()` checks all 30 chunks against the question's rules. A handful
   get a grade; every other chunk is 0.
3. **Look up the grade of each search result.** The search returns its top 10, best first. Each result
   is worth points: grade 3 is worth 7, grade 2 is worth 3, grade 1 is worth 1. The points shrink the
   lower the result sits: divided by 1 at rank 1, by 1.58 at rank 2, by 2.81 at rank 6.
4. **Compare with the best possible order.** Add up the points the search earned, and the points the
   same graded chunks would earn in the perfect order. nDCG@10 is the first divided by the second, so
   1.0 always means "best answers on top", whatever the question.

**What you should see.** For `R02`: three graded chunks; the grade-3 chunk at rank 6; 2.92 points
earned out of 9.39 possible; nDCG@10 = 0.311, the same number as in the table above. For `R17`: no
chunk matches its rules at all, because its answer is in the scanned PDF that this track cannot read,
so the score is 0.000 and no search change can help. For `R13`: the answer is at rank 2, which gives
0.631.

**Why this step is here.** Every number in this lab is an average of 20 scores built exactly this
way. Only step 3 changes from stage to stage, because only the search changes. The answer key stays
fixed, so a change in the score can only come from the search.

In [ ]:
# YOUR CALL: which question to open up. Try R02, then R17, then R13.
QID = "R02"

import math, re
item = next(it for it in golden if it["id"] == QID)
print(f'{item["id"]}: "{item["query"]}"')

print("\n1. The answer key: the rules a person wrote for this question")
for rule in item["rules"]:
    print(f'   grade {rule["grade"]}   any chunk from {rule["source"]} containing "{rule["contains"]}"')

rel = relevance(item, corpus.chunks)                 # {chunk id: grade}, only the graded ones
if rel:
    print(f"\n2. Those rules grade {len(rel)} of the {len(corpus.chunks)} chunks. Every other chunk is grade 0.")
else:
    print(f"\n2. Those rules match none of the {len(corpus.chunks)} chunks: the text they look for never reached the index.")

print("\n3. What meaning search returned, with each result's grade and points")
hits = corpus.index.search(item["query"], k=10)
earned = 0.0
for rank, (_, ch) in enumerate(hits, start=1):
    grade = rel.get(ch.id, 0)
    points = (2 ** grade - 1) / math.log2(rank + 1)  # grade into points, shrunk by position
    earned += points
    lines = [l for l in re.sub(r"^\[[^\]]*\]\s*", "", ch.text).split("\n") if len(l.strip()) > 12]
    heading = (lines[0] if lines else ch.text).replace(chr(0x2014), "-")[:40]
    print(f"   rank {rank:2d}   grade {grade}   points {points:4.2f}   {ch.meta['source'][:30]:30s} {heading}")
found = {ch.id for _, ch in hits}
left_out = sorted((g for cid, g in rel.items() if cid not in found), reverse=True)
if left_out:
    print(f"   graded chunks outside the top 10, worth nothing here: grades {left_out}")

print("\n4. The score")
ideal = sorted(rel.values(), reverse=True)[:10]
best = sum((2 ** g - 1) / math.log2(r + 1) for r, g in enumerate(ideal, start=1))
if best == 0:
    print("   No chunk matches this question's rules, so there is nothing to find: nDCG@10 = 0.000")
else:
    print(f"   points this search earned          {earned:.2f}")
    print(f"   points the best order would earn   {best:.2f}   (grades {ideal}, best first, from rank 1)")
    print(f"   nDCG@10 = {earned:.2f} / {best:.2f} = {earned / best:.3f}")

## Step 3. Add keyword search and fuse the two lists

**What you will do.** Build the hybrid searcher. It runs meaning search and BM25 keyword search over
the same pieces of text, then merges the two ranked lists with reciprocal rank fusion, which scores
each result by its position in each list rather than by the raw scores, because the two kinds of score
are not comparable. In the `YOUR CALL` cell, set how many candidates each of the two searches
contributes before the merge, and write down whether you expect a gain.

**What you should see.** The average moves from 0.803 to 0.806, a gain of 0.003. That is almost
nothing. Individual questions barely move either.

**Why this step is here.** Hybrid search is the standard advice, and on this corpus the standard
advice does close to nothing. The corpus is small and the questions are worded much like the
documents, so keyword search finds what meaning search already found. On a corpus full of product
codes, part numbers and error strings you would expect a much larger gain. The lesson is not that
hybrid is useless: it is that you measure it on your own data before you pay for it.

In [ ]:
# YOUR CALL: candidates taken from each of the two searches before the merge
K_EACH = 50
# YOUR CALL: do you expect the average to rise by more than 0.05? True or False
PREDICT_BIG_GAIN = False

In [ ]:
# Hybrid wraps the existing index: it runs keyword search alongside meaning search and
# merges the two lists by position, because their scores are not comparable.
hyb = Hybrid(corpus.index)
hybrid = stage_scores(lambda q: hyb.search(q, k=50, k_each=K_EACH))
mean_hybrid = statistics.mean(hybrid)
print(f"meaning search only : {mean_dense:.3f}")
print(f"with keyword search : {mean_hybrid:.3f}")
print(f"gain                : {mean_hybrid - mean_dense:+.3f}")
print("you predicted a gain above 0.05:", PREDICT_BIG_GAIN,
      "->", "correct" if PREDICT_BIG_GAIN == (mean_hybrid - mean_dense > 0.05) else "not this time")

## Step 4. Rerank the shortlist, then look at what got worse

**What you will do.** Take the merged shortlist and reorder it with a reranker, which scores each
candidate against the question directly instead of comparing vectors. Measure again, then print every
question whose score went down.

**What you should see.** The average rises from 0.806 to 0.815. One question improves a lot: R13,
"refund rate in Q1 2024", goes from 0.631 to 1.000. Three questions get worse: R02 from 0.31 to 0.26,
R04 from 0.82 to 0.79, and R05 from 0.71 to 0.59.

**Why this step is here.** Two reasons. First, a change that helps on average can still hurt specific
questions, and the list of what got worse is the first thing a careful architect asks for. Second, the
reranker running here is a stand-in built from word statistics, not the real cross-encoder model,
because this track runs with no keys. Report what you measure. The Azure track at the end of the
notebook runs the same questions against a real reranker so you can see the difference.

In [ ]:
# Same merged list as before, then reordered. idf carries the word statistics the
# offline reranker uses to decide which candidate really answers the question.
reranked = stage_scores(lambda q: rerank(q, hyb.search(q, k=50, k_each=K_EACH)[:50], k=10, idf=hyb.idf))
mean_rerank = statistics.mean(reranked)
print(f"with keyword search : {mean_hybrid:.3f}")
print(f"after reranking     : {mean_rerank:.3f}   ({mean_rerank - mean_hybrid:+.3f})")

print("\nquestions that got WORSE after reranking:")
worse = [(it["id"], it["query"], h, r) for it, h, r in zip(golden, hybrid, reranked) if r < h - 1e-9]
for qid, q, h, r in worse:
    print(f"  {qid}  {q[:46]:46s} {h:.2f} -> {r:.2f}")
if not worse:
    print("  none this run")
print("\npick one of those and be ready to say why in the debrief.")

### Why three questions got worse after reranking

All three get worse for the same reason: the reranker in this notebook is a stand-in that does not read
for meaning. It only counts which words of the question appear in each chunk.

**How the stand-in scores a chunk** (`rerank` in `src/retrieval.py`):

- 70 percent comes from shared words: how many of the question's words appear in the chunk, with rare
  words counting more, plus a small bonus when those words sit close together.
- 30 percent comes from where the chunk already sat in the merged list, so it nudges the order rather
  than rebuilding it.

Nothing in that checks whether the chunk answers the question. A chunk that happens to repeat the
question's words gets pushed up.

| Question | Before, after | What happened |
|---|---|---|
| R02 "who pays when food is spilled in transit" | 0.311, 0.262 | The answer says "packaging inadequate for transport of liquid contents", which shares almost no words with the question. A support-rules row about a "food safety complaint" shares "food" and moves up. The answer drops from 6th to 9th. |
| R05 "what is the first response target for a spillage complaint" | 0.710, 0.590 | The answer is one row inside a long support-rules table. The table's header chunk repeats "first response" and "complaint" close together, so it scores higher. The answer slips from 2nd to 3rd. |
| R04 "when must an agent escalate to a supervisor" | 0.821, 0.794 | The best answer is in the scanned PDF, which this track cannot read, so only weaker chunks compete. Two spillage sections that share question words move above a grade-1 support-rules row, which drops from 3rd to 5th. |

**What to take into the debrief.** Shared words are evidence that a chunk is about the same topic, not
that it answers the question. A real reranker reads the question and the chunk together and can tell the
difference. In Lab 3b, with a real model doing the reranking, no question got worse.

## Step 5. Drop the documents that are no longer in force

**What you will do.** Add the recency filter. It uses one label you attached back in Lab 1,
`superseded`, and removes every piece of text from a withdrawn document before ranking. Measure the
same 20 questions again.

**What you should see.** The average rises from 0.815 to 0.874. That is the largest single move in
this lab, larger than keyword search and reranking put together. R01, the question from Step 1, goes
from 0.58 to 0.92. R02 stays near 0.33 and is still failing.

**Why this step is here.** The two clever retrieval changes cost you a second search, a merge and a
reranking pass, and between them they bought 0.012. One label from ingestion bought 0.059. When
someone proposes a new retrieval technique, this is the comparison to make them run first.

In [ ]:
# The only change here is where=current_only, which removes pieces from withdrawn
# documents during the search rather than after it.
filtered = stage_scores(
    lambda q: rerank(q, hyb.search(q, k=50, k_each=K_EACH, where=current_only)[:50], k=10, idf=hyb.idf))
mean_filter = statistics.mean(filtered)
print(f"after reranking      : {mean_rerank:.3f}")
print(f"after recency filter : {mean_filter:.3f}   ({mean_filter - mean_rerank:+.3f})")

print("\nthe five Week 1 failures, start to finish:")
for it, d, f in zip(golden, dense, filtered):
    if it.get("week1_failure"):
        verdict = "still failing" if f < 0.5 else "improved" if f - d >= 0.1 else "no change"
        print(f"  {it['id']}  {it['query'][:46]:46s} {d:.2f} -> {f:.2f}  {verdict}")

## Step 5b. Follow two questions through all four stages

**What you will do.** Run the next cell. It takes two of the Week 1 failures, R01 and R02, and for
each stage prints where their graded chunks landed, the points those positions earn, and the score.
You can add any other question id to `FOLLOW`.

**What you should see.**

| Stage | R01 | R02 |
|---|---|---|
| meaning only | 0.579 | 0.311 |
| + keyword | 0.579 | 0.311 |
| + rerank | 0.579 | 0.262 |
| + recency | 0.917 | 0.333 |

**What is happening to R01**, "how many refund claims can a customer make in thirty days". The
withdrawn 2022 policy says "up to two refund claims", the 2024 policy says "up to three". The two
sentences are identical except for that one word. Meaning search, keyword search and the stand-in
reranker all measure how alike the words are, so all three put the withdrawn chunk first and the right
one second: 0.579 at every stage. Only the recency filter can tell them apart, because it reads the
`superseded` label you attached in Lab 1, not the text. With the old chunk gone, the right one is first:
0.917. It is not 1.000 because the grade-1 chunk, a note in the claims spreadsheet, never reaches the
top 10.

**What is happening to R02**, "who pays when food is spilled in transit". The answer is in the partner
agreement, worded "packaging inadequate for transport of liquid contents". It shares almost no words
with the question. Meaning search puts it 6th, keyword search 5th, and the merge keeps it 6th: 0.311.
The stand-in reranker rewards shared words, so it pushes the answer down to 9th: 0.262. The recency
filter removes withdrawn chunks that were crowding the top, which lifts a grade-1 chunk to 2nd, but
the answer stays 6th: 0.333. Still failing.

**Why this step is here.** The two questions fail for different reasons. R01 is a data problem: the
right fact is there but competes with a withdrawn one, and a label fixes it. R02 is a vocabulary
problem: every stage on this track works with words, and the question and the answer use different
ones. Only a model that understands meaning fixes it; with real models (Lab 3b) R02 scores 0.88 on
meaning search alone. Before you pick a technique, find out which kind of failure you have.

In [ ]:
# YOUR CALL: which questions to follow. Add any id from the table in Step 2.
FOLLOW = ["R01", "R02"]

import math
STAGE_RUNS = {
    "1 meaning only": lambda q: corpus.index.search(q, k=50),
    "2 + keyword":    lambda q: hyb.search(q, k=50, k_each=K_EACH),
    "3 + rerank":     lambda q: rerank(q, hyb.search(q, k=50, k_each=K_EACH)[:50], k=10, idf=hyb.idf),
    "4 + recency":    lambda q: rerank(q, hyb.search(q, k=50, k_each=K_EACH, where=current_only)[:50], k=10, idf=hyb.idf),
}
for qid in FOLLOW:
    item = next(it for it in golden if it["id"] == qid)
    rel = relevance(item, corpus.chunks)
    best = sum((2 ** g - 1) / math.log2(r + 1) for r, g in enumerate(sorted(rel.values(), reverse=True)[:10], start=1))
    print(f'\n{qid}: "{item["query"]}"   best possible points: {best:.2f}')
    for name, run in STAGE_RUNS.items():
        ids = [ch.id for _, ch in run(item["query"])]
        parts, earned = [], 0.0
        for grade in sorted(set(rel.values()), reverse=True):
            for cid, g in rel.items():
                if g != grade:
                    continue
                rank = ids.index(cid) + 1 if cid in ids else None
                if rank and rank <= 10:
                    pts = (2 ** g - 1) / math.log2(rank + 1)
                    earned += pts
                    parts.append(f"grade {g} at rank {rank} ({pts:.2f})")
                else:
                    parts.append(f"grade {g} outside the top 10 (0)")
        score = earned / best if best else 0.0
        print(f"   {name:15s} {', '.join(parts) or 'nothing graded'}   ->  {earned:.2f} / {best:.2f} = {score:.3f}")

## Step 6. The four stages side by side

**What you will do.** Run the last cell. It measures all four stages again, this time timing each one,
and prints the table you will hand in.

**What you should see.** Average nDCG@10 of 0.803, 0.806, 0.815 and 0.874 for the four stages, and a
time per question that grows from roughly 0.2 milliseconds to roughly 0.7 milliseconds. Your times
will differ from this machine, the scores will not.

**Why this step is here.** This table is the deliverable. It says what each stage bought and what each
stage cost, which is the only honest way to decide what to keep in production. Copy it into the **Your write-up** cell at the bottom of this notebook, with your answer to the one question that matters: which of the three
additions would you ship, and which would you drop.

In [ ]:
import time

stages = {
    "1 meaning only": lambda q: corpus.index.search(q, k=50),
    "2 + keyword":    lambda q: hyb.search(q, k=50, k_each=K_EACH),
    "3 + rerank":     lambda q: rerank(q, hyb.search(q, k=50, k_each=K_EACH)[:50], k=10, idf=hyb.idf),
    "4 + recency":    lambda q: rerank(q, hyb.search(q, k=50, k_each=K_EACH, where=current_only)[:50],
                                       k=10, idf=hyb.idf),
}
# Run every stage again, this time with a clock around the search itself, so the table
# reports what each stage bought and what it cost.
summary = []
for name, fn in stages.items():
    scores, times = [], []
    for item in golden:
        rel = relevance(item, corpus.chunks)
        t0 = time.perf_counter()
        ids = [ch.id for _, ch in fn(item["query"])]
        times.append((time.perf_counter() - t0) * 1000)
        scores.append(ndcg_at_k(ids, rel, k=10))
    summary.append({"stage": name, "mean nDCG@10": round(statistics.mean(scores), 3),
                    "p50 ms": round(statistics.median(times), 2),
                    "p95 ms": round(sorted(times)[int(0.95 * len(times)) - 1], 2)})
banner("what each stage bought, and what it cost")
print(table(summary, ["stage", "mean nDCG@10", "p50 ms", "p95 ms"]))

## When you are done

- Fill in the **Your write-up** cell at the bottom of this notebook. It asks for the four stage scores, the query that got worse and
  your theory, and the one change you would ship.
- The same four stages with real models: run `databricks/lab03b_real_models` on Databricks.
- Extra challenge: add one question of your own to `data/golden/retrieval_golden.json`, with its
  grading rules, and see whether your four numbers hold.

## Your write-up

**Type your answers into this cell.** Double click it to edit, then click away to render it again.
Everything you write here leaves with the notebook when you export it, and that export is what you
hand in.

Do not write in `handouts/lab3_answers.md`. It is the blank answer sheet, and a fresh copy of the repository replaces it.

---

**Pair:**

### 1. What each stage bought, and what it cost

| Stage | mean nDCG@10 | p50 ms | p95 ms |
|---|---|---|---|
| meaning search only | | | |
| + keyword search | | | |
| + reranking | | | |
| + recency filter | | | |

### 2. The Week 1 failures

| Query id | Before | After | Fixed? |
|---|---|---|---|
| R01 | | | |
| R02 | | | |
| R04 | | | |
| R10 | | | |
| R20 | | | |

Which one is still failing, and what would it take to fix it?

### 3. One query that got worse after reranking

Query id: ___  Score before: ___  after: ___

Your theory, in two sentences:

### 4. The decision

Of the three additions, which would you ship, and which would you drop? Give the number
that decides it, not an opinion.

### 5. What you would measure next